# LAND model results

Visualize Optuna trials, training histories, predictions, ensemble uncertainty, and wet/dry performance from saved pipeline artifacts.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FREQ = "weekly"
RUN_NAME = "land_weekly_gamma_mse_cv3both_n100_lowlr"
STUDY_NAME = RUN_NAME
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Daily_Modeling").exists())
OUTPUT_DIR = PROJECT_DIR / "Daily_Modeling" / "output" / FREQ
RUN_DIR = OUTPUT_DIR / "results" / RUN_NAME
TUNING_DIR = OUTPUT_DIR / "tuning" / STUDY_NAME
RUN_DIR

## Hyperparameter tuning

In [ ]:
trials_path = TUNING_DIR / "all_trials.csv"
if trials_path.exists():
    trials = pd.read_csv(trials_path).sort_values("trial")
    display(trials.nsmallest(10, "value"))
    ax = trials.plot(x="trial", y="value", marker=".", figsize=(10, 4), legend=False)
    ax.plot(trials["trial"], trials["value"].cummin(), label="Best so far")
    ax.set(ylabel="Objective", title="Optimization history")
    ax.legend()
else:
    print(f"No tuning table found at {trials_path}")

## Training histories

In [ ]:
history_paths = sorted(RUN_DIR.glob("**/training_history_seed*.json"))
fig, ax = plt.subplots(figsize=(10, 5))
for path in history_paths:
    history = json.loads(path.read_text())
    label = str(path.relative_to(RUN_DIR)).replace("training_history_", "").replace(".json", "")
    ax.plot(history["val_loss"], alpha=0.7, label=label)
ax.set(xlabel="Epoch", ylabel="Validation loss", title="Validation histories")
if history_paths:
    ax.legend(fontsize=7, ncol=2)
else:
    print("No JSON histories found. Re-run training with the current pipeline to generate them.")
fig.tight_layout()

## Predictions and uncertainty

In [ ]:
prediction_paths = sorted((RUN_DIR / "inference").glob("predictions_*.npz"))
prediction_paths

In [ ]:
PREDICTION_FILE = RUN_DIR / "inference" / "predictions_test_all.npz"
with np.load(PREDICTION_FILE, allow_pickle=True) as raw:
    pred = {key: raw[key] for key in raw.files}

y_true = pred["y_true"].reshape(-1)
y_pred = pred["y_pred_mean"].reshape(-1)
y_std = pred["y_pred_std"].reshape(-1)
limit = float(max(y_true.max(), y_pred.max()))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_true, y_pred, s=10, alpha=0.35)
axes[0].plot([0, limit], [0, limit], "k--")
axes[0].set(xlabel="Observed (mm)", ylabel="Predicted (mm)", title="Observed vs predicted")
axes[1].scatter(y_pred, y_std, s=10, alpha=0.35)
axes[1].set(xlabel="Ensemble mean (mm)", ylabel="Ensemble std (mm)", title="Predictive uncertainty")
fig.tight_layout()

## Wet/dry evaluation

In [ ]:
WET_THRESHOLD_MM = 1.0
obs_wet = y_true >= WET_THRESHOLD_MM
pred_wet = y_pred >= WET_THRESHOLD_MM
confusion = pd.DataFrame(
    [[np.sum(~obs_wet & ~pred_wet), np.sum(~obs_wet & pred_wet)],
     [np.sum(obs_wet & ~pred_wet), np.sum(obs_wet & pred_wet)]],
    index=["Observed dry", "Observed wet"],
    columns=["Predicted dry", "Predicted wet"],
)
display(confusion)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(confusion, cmap="Blues")
for row in range(2):
    for col in range(2):
        ax.text(col, row, f"{confusion.iloc[row, col]:,}", ha="center", va="center")
ax.set(xticks=[0, 1], xticklabels=confusion.columns, yticks=[0, 1], yticklabels=confusion.index, title=f"Wet/dry confusion ({WET_THRESHOLD_MM:g} mm)")
fig.colorbar(image, ax=ax)
fig.tight_layout()

## Metrics by output split

In [ ]:
metric_rows = []
for path in sorted((RUN_DIR / "inference").glob("metrics_*.json")):
    metric_rows.append({"split": path.stem.removeprefix("metrics_"), **json.loads(path.read_text())})
pd.DataFrame(metric_rows).set_index("split") if metric_rows else pd.DataFrame()